# 03. Case Retrieval System

This notebook implements the Case Retrieval engine for our Case-Based Reasoning (CBR) system. It reads the structured case metadata, builds TF-IDF vector representations of the text, trains an SVM model to classify/predict the case outcomes (verdict), and implements a Cosine Similarity retrieval mechanism to search for the top-k most similar historical cases given a new query.

### Tasks:
1. **Read Cases**: Load `data/processed/cases.csv` and associate them with cleaned texts from `data/raw_text/`.
2. **Vectorization**: Build TF-IDF representations of the case texts.
3. **Train Classifier**: Split dataset 80:20 into train/test sets and train an SVM classifier to predict case outcomes.
4. **Save Model Artifacts**: Persist the trained SVM model and TF-IDF vectorizer to `models/`.
5. **Implement Retrieval**: Define the `retrieve(query, k=5)` function using cosine similarity.
6. **Display Results**: Query the system and display the top-5 retrieved cases.

In [ ]:
import os
import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

# Import Sastrawi components for query preprocessing
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [ ]:
# Define paths
PROCESSED_CSV_PATH = Path("../data/processed/cases.csv")
RAW_TEXT_DIR = Path("../data/raw_text")
MODELS_DIR = Path("../models")

MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load cases and merge with preprocessed full text
df_cases = pd.read_csv(PROCESSED_CSV_PATH)

texts = []
for _, row in df_cases.iterrows():
    case_id = row['Case ID']
    txt_file = RAW_TEXT_DIR / f"{case_id}.txt"
    
    if txt_file.exists():
        with open(txt_file, 'r', encoding='utf-8') as f:
            texts.append(f.read())
    else:
        # Fallback to combined metadata fields
        combined_meta = f"{row['Ringkasan fakta']} {row['Amar putusan']} {row['Pasal']}"
        texts.append(combined_meta)

df_cases['cleaned_text'] = texts
print(f"Loaded {len(df_cases)} cases for vectorization.")

In [ ]:
# Define classification labels (Verdict: Conviction/Bersalah vs Acquittal/Bebas/Lepas)
def label_verdict(row):
    amar = str(row['Amar putusan']).lower()
    if "bebas" in amar or "lepas" in amar or "onslag" in amar or "tidak terbukti" in amar:
        return 0  # Acquittal / Bebas / Lepas
    return 1  # Conviction / Bersalah

df_cases['Label'] = df_cases.apply(label_verdict, axis=1)

print("Verdict Distribution:")
print(df_cases['Label'].value_counts())

# Split dataset 80:20
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df_cases['cleaned_text'], 
    df_cases['Label'], 
    test_size=0.2, 
    random_state=42,
    stratify=df_cases['Label']
)
print(f"Train size: {len(X_train_raw)}, Test size: {len(X_test_raw)}")

In [ ]:
# TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=1000)
X_train_tfidf = vectorizer.fit_transform(X_train_raw)
X_test_tfidf = vectorizer.transform(X_test_raw)

# Also fit vectorizer on full dataset for retrieval matrices
vectorizer_full = TfidfVectorizer(max_features=1000)
tfidf_matrix_full = vectorizer_full.fit_transform(df_cases['cleaned_text'])

print(f"TF-IDF vocabulary size: {len(vectorizer_full.get_feature_names_out())}")

In [ ]:
# Train SVM Classifier
svm_classifier = SVC(kernel='linear', probability=True, random_state=42)
svm_classifier.fit(X_train_tfidf, y_train)

# Evaluate model
y_pred = svm_classifier.predict(X_test_tfidf)
print("SVM Model Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Save Model and Vectorizer
model_path = MODELS_DIR / "svm_model.pkl"
vectorizer_path = MODELS_DIR / "tfidf_vectorizer.pkl"

with open(model_path, 'wb') as f:
    pickle.dump(svm_classifier, f)

with open(vectorizer_path, 'wb') as f:
    # Save the vectorizer fitted on the full corpus for retrieval purposes
    pickle.dump(vectorizer_full, f)

print(f"Saved SVM Model to {model_path.resolve()}")
print(f"Saved TF-IDF Vectorizer to {vectorizer_path.resolve()}")

In [ ]:
# Setup Sastrawi components for query cleaning inside the retrieve function
stop_factory = StopWordRemoverFactory()
indonesian_stopwords = set(stop_factory.get_stop_words())
custom_stopwords = {'dan', 'yang', 'untuk', 'dengan', 'pada', 'atau', 'adalah', 'bahwa'}
indonesian_stopwords.update(custom_stopwords)

stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

def preprocess_query(query: str) -> str:
    """Preprocesses input queries using the same pipeline as the case base."""
    # Lowercase & remove punctuation/numbers
    query_clean = re.sub(r'[^a-zA-Z\s]', ' ', query.lower())
    # Tokenize
    tokens = query_clean.split()
    # Stopword removal
    filtered_tokens = [w for w in tokens if w not in indonesian_stopwords]
    # Stemming
    stemmed_tokens = [stemmer.stem(w) for w in filtered_tokens]
    return ' '.join([w for w in stemmed_tokens if len(w) > 1])

def retrieve(query: str, k: int = 5) -> pd.DataFrame:
    """
    Retrieves the top-k most similar cases from the case base using cosine similarity.
    """
    # Preprocess query
    cleaned_query = preprocess_query(query)
    print(f"Original query: '{query}'")
    print(f"Cleaned & stemmed query: '{cleaned_query}'")
    
    # Vectorize query
    query_vector = vectorizer_full.transform([cleaned_query])
    
    # Compute Cosine Similarity
    similarities = cosine_similarity(query_vector, tfidf_matrix_full).flatten()
    
    # Combine with case database
    df_res = df_cases.copy()
    df_res['Similarity Score'] = similarities
    
    # Sort and pick top k
    top_k_cases = df_res.sort_values(by='Similarity Score', ascending=False).head(k)
    
    return top_k_cases[[
        'Case ID', 
        'Nomor perkara', 
        'Nama terdakwa', 
        'Pasal', 
        'Similarity Score', 
        'Amar putusan'
    ]]

In [ ]:
# Test Retrieval Engine
sample_query = "pemalsuan tanda tangan lurah dalam surat tanah ganti rugi materai palsu"
results = retrieve(sample_query, k=5)

print("\nTop-5 Retrieved Cases:")
display(results)